# Initial Libraies

In [1]:
!pip install job-shop-lib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.1/293.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 8.4 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from job_shop_lib.visualization.gantt import plot_gantt_chart
from job_shop_lib import JobShopInstance, Operation
from job_shop_lib.dispatching import (
    ReadyOperationsFilterType,
    create_composite_operation_filter,
    get_breakdown_calculator,
    Dispatcher
)
from job_shop_lib.dispatching.rules import (
    DispatchingRuleSolver,
    DispatchingRuleType,
)

from job_shop_lib.benchmarking import load_benchmark_instance



from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List
from tqdm import tqdm

In [3]:
plt.style.use("ggplot")

In [4]:
def plot_gantt_chart_from_dispatching_rule(
    dispatching_rule: DispatchingRuleType,
    instance: JobShopInstance,
    optimization_level: int = 0,
) -> plt.Figure:

    if optimization_level == 0:
        pruning_function = None
    elif optimization_level == 1:
        pruning_function = ReadyOperationsFilterType.DOMINATED_OPERATIONS
    elif optimization_level == 2:
        pruning_function = create_composite_operation_filter(
            [
                ReadyOperationsFilterType.DOMINATED_OPERATIONS,
                ReadyOperationsFilterType.NON_IMMEDIATE_MACHINES,
            ]
        )
    else:
        raise ValueError(f"Invalid optimization level: {optimization_level}")

    solver = DispatchingRuleSolver(
        dispatching_rule,
        ready_operations_filter=pruning_function,
    )
    solution = solver.solve(instance)

    title = f"{instance.name} - {dispatching_rule.name} (optimization level {optimization_level})"
    number_of_x_ticks = 10 if solution.makespan() >= 1000 else 15
    fig, ax = plot_gantt_chart(
        solution, title=title, number_of_x_ticks=number_of_x_ticks
    )
    if instance.num_jobs > 20:
        # Remove legend if there are too many jobs
        ax.legend().remove()
    return fig

In [5]:
class JSPMUInstance:

    def __init__(self, name=""):
        self.name = name
        self.n_jobs = 0
        self.n_machines = 0

        # jobs[j] = [(machine, duration), ...]
        self.jobs = []

        # machine_holes[m] = [(start, end), ...]
        self.machine_holes = {}

    @staticmethod
    def read(filepath):

        instance = JSPMUInstance(Path(filepath).stem)

        with open(filepath, "r") as f:
            lines = [l.strip() for l in f if l.strip()]

        # ---- header ----
        header = lines[0].split()
        instance.n_jobs = int(header[0])
        instance.n_machines = int(header[1])

        # ---- jobs ----
        for j in range(instance.n_jobs):

            tokens = list(map(int, lines[1 + j].split()))

            if len(tokens) != 2 * instance.n_machines:
                raise ValueError(
                    f"Job {j} has wrong number of entries."
                )

            ops = []

            for k in range(0, len(tokens), 2):

                machine = tokens[k]
                duration = tokens[k + 1]

                if machine < 0 or machine >= instance.n_machines:
                    raise ValueError(f"Invalid machine {machine}")

                ops.append((machine, duration))

            instance.jobs.append(ops)

        # ---- initialize empty holes ----
        for m in range(instance.n_machines):
            instance.machine_holes[m] = []

        # ---- holes section ----
        idx = 1 + instance.n_jobs

        if idx < len(lines):

            if lines[idx] != "[MACHINE_HOLES]":
                raise ValueError("Missing [MACHINE_HOLES] section")

            idx += 1

            while idx < len(lines):

                tokens = list(map(int, lines[idx].split()))

                machine = tokens[0]
                n_holes = tokens[1]

                expected = 2 + 2 * n_holes
                if len(tokens) != expected:
                    raise ValueError(
                        f"Wrong hole format for machine {machine}"
                    )

                holes = []

                pos = 2
                for _ in range(n_holes):

                    start = tokens[pos]
                    duration = tokens[pos + 1]

                    if duration <= 0:
                        raise ValueError("Hole duration must be positive")

                    holes.append((start, start + duration))

                    pos += 2

                instance.machine_holes[machine] = holes

                idx += 1

        return instance

    def __str__(self) -> str:
        print('Name:',self.name)
        print("n jobs:",self.n_jobs)
        print("n machines:",self.n_machines)
        print("jobs:")
        for j in self.jobs:
          print(j)
        print("Unavailabilities:")
        for h in self.machine_holes:
          print('Machine',h,':',self.machine_holes[h])
        return ''

# schedule with job shop lib and then correct unavailabilities

In [6]:
def to_job_shop_lib_instance(jspmu_instance, name=None):
    """
    Convert a JSPMUInstance into a job_shop_lib.JobShopInstance.

    Conventions:
    - Original jobs keep their original routing.
    - Each machine unavailability (start, end) becomes a dummy one-operation job.
    - metadata["release_dates"]:
        * 0 for original jobs
        * hole start for dummy maintenance jobs
    - metadata["deadlines"]:
        * float("inf") for original jobs
        * hole end for dummy maintenance jobs
    """

    jobs = []
    release_dates = []
    deadlines = []

    # --- Original jobs ---
    # jspmu_instance.jobs[j] = [(machine, duration), ...]
    for job in jspmu_instance.jobs:
        converted_job = [Operation(machine, duration) for machine, duration in job]
        jobs.append(converted_job)
        release_dates.append(0)
        deadlines.append(float("inf"))

    # --- Dummy jobs for machine holes ---
    # jspmu_instance.machine_holes[m] = [(start, end), ...]
    # Each hole becomes a 1-operation job of duration end-start on machine m
    for machine in sorted(jspmu_instance.machine_holes):
        holes = jspmu_instance.machine_holes[machine]
        for start, end in holes:
            duration = end - start
            if duration <= 0:
                raise ValueError(
                    f"Invalid hole on machine {machine}: ({start}, {end})"
                )

            dummy_job = [Operation(machine, duration)]
            jobs.append(dummy_job)
            release_dates.append(start)
            deadlines.append(end)

    instance = JobShopInstance(
        jobs=jobs,
        name=name if name is not None else jspmu_instance.name
    )

    instance.metadata["release_dates"] = release_dates
    instance.metadata["deadlines"] = deadlines

    return instance

In [7]:
def to_job_shop_lib_instance(jspmu_instance, name=None):
    """
    Convert a JSPMUInstance into a JobShopInstance with only real jobs.
    Machine holes are enforced later through the Dispatcher.
    """
    jobs = []

    for job in jspmu_instance.jobs:
        converted_job = [Operation(machine, duration) for machine, duration in job]
        jobs.append(converted_job)

    return JobShopInstance(
        jobs=jobs,
        name=name if name is not None else jspmu_instance.name,
    )

In [8]:
def solve_jspmu_with_dispatching(jspmu_instance, dispatching_rule="most_work_remaining"):
    """
    Solve JSPMU with a dispatching rule while enforcing machine holes
    as machine downtimes via a custom Dispatcher.
    """
    jsl_inst = to_job_shop_lib_instance(jspmu_instance)

    # IMPORTANT:
    # JobShopLib breakdowns must be (start_time, duration), not (start, end).
    breakdowns = {
        m: [(start, end - start) for start, end in jspmu_instance.machine_holes.get(m, [])]
        for m in range(jspmu_instance.n_machines)
    }

    dispatcher = Dispatcher(
        instance=jsl_inst,
        start_time_calculator=get_breakdown_calculator(breakdowns),
    )

    solver = DispatchingRuleSolver(
        dispatching_rule=dispatching_rule,
    )

    schedule = solver.solve(jsl_inst, dispatcher=dispatcher)
    return jsl_inst, dispatcher, schedule

In [9]:
def check_breakdowns(jspmu_instance):
    for m in range(jspmu_instance.n_machines):
        for start, end in jspmu_instance.machine_holes.get(m, []):
            if end <= start:
                raise ValueError(f"Invalid hole on machine {m}: ({start}, {end})")

In [10]:
def validate_jsl_schedule_against_jspmu_real_only(jspmu_instance, schedule):
    """
    Validate a JobShopLib schedule against the original JSPMU instance
    when the schedule contains only the real jobs.
    """
    violations = {
        "machine_overlap": [],
        "job_precedence": [],
        "hole_overlap": [],
    }

    machine_ops = {m: [] for m in range(jspmu_instance.n_machines)}
    for m, ops in enumerate(schedule.schedule):
        for sop in ops:
            machine_ops[m].append(sop)

    # Machine overlap
    for m in range(jspmu_instance.n_machines):
        ops = sorted(machine_ops[m], key=lambda o: o.start_time)
        for a, b in zip(ops, ops[1:]):
            if a.end_time > b.start_time:
                violations["machine_overlap"].append(
                    (m, (a.job_id, a.position_in_job, a.start_time, a.end_time),
                        (b.job_id, b.position_in_job, b.start_time, b.end_time))
                )

    # Job precedence
    by_job = {}
    for m in range(jspmu_instance.n_machines):
        for sop in machine_ops[m]:
            by_job.setdefault(sop.job_id, []).append(sop)

    for j in range(jspmu_instance.n_jobs):
        ops = sorted(by_job.get(j, []), key=lambda o: o.position_in_job)
        for a, b in zip(ops, ops[1:]):
            if a.end_time > b.start_time:
                violations["job_precedence"].append(
                    (j,
                     (a.position_in_job, a.start_time, a.end_time),
                     (b.position_in_job, b.start_time, b.end_time))
                )

    # Hole overlap
    for m in range(jspmu_instance.n_machines):
        holes = jspmu_instance.machine_holes.get(m, [])
        for sop in machine_ops[m]:
            s, e = sop.start_time, sop.end_time
            for hs, he in holes:
                if max(s, hs) < min(e, he):
                    violations["hole_overlap"].append(
                        (m, sop.job_id, sop.position_in_job, s, e, hs, he)
                    )

    feasible = all(len(v) == 0 for v in violations.values())
    return feasible, violations

In [11]:
def schedule_to_plot_inputs(schedule, n_original_jobs=None):
    """
    Convert a job_shop_lib Schedule object into:
        start_times[(job_id, op_id)]
        end_times[(job_id, op_id)]
        makespan
    """

    # unwrap Schedule object
    if hasattr(schedule, "schedule"):
        machine_sequences = schedule.schedule
    elif hasattr(schedule, "_schedule"):
        machine_sequences = schedule._schedule
    else:
        raise TypeError(
            "Could not find the machine-wise operation lists inside `schedule`. "
            f"Available attributes: {dir(schedule)}"
        )

    start_times = {}
    end_times = {}
    makespan = 0

    for machine_ops in machine_sequences:
        for sched_op in machine_ops:
            op = sched_op.operation

            j = op.job_id
            k = op.position_in_job
            s = sched_op.start_time
            e = s + op.duration

            if n_original_jobs is not None and j >= n_original_jobs:
                continue

            start_times[(j, k)] = s
            end_times[(j, k)] = e
            makespan = max(makespan, e)

    return start_times, end_times, makespan

# Import instances

In [12]:
import zipfile
import os

In [14]:
zip_path = "generated_jspmu_instances_litarature_2.zip"
extract_dir = "generated_jspmu_instances_litarature"

# List to store file names inside the zip
file_names = []

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    file_names = zip_ref.namelist()  # names of all files/folders inside the zip
    zip_ref.extractall(extract_dir)  # unzip everything

print("Files inside zip:")
print(file_names)

Files inside zip:
['abz5_mu_mtbf300_durlognorm35_03.jspmu', 'abz6_mu_mtbf300_durlognorm35_03.jspmu', 'ft06_mu_mtbf300_durlognorm35_03.jspmu', 'ft10_mu_mtbf300_durlognorm35_03.jspmu', 'la01_mu_mtbf300_durlognorm35_03.jspmu', 'la02_mu_mtbf300_durlognorm35_03.jspmu', 'la03_mu_mtbf300_durlognorm35_03.jspmu', 'la04_mu_mtbf300_durlognorm35_03.jspmu', 'la05_mu_mtbf300_durlognorm35_03.jspmu', 'la16_mu_mtbf300_durlognorm35_03.jspmu', 'la17_mu_mtbf300_durlognorm35_03.jspmu', 'la18_mu_mtbf300_durlognorm35_03.jspmu', 'la19_mu_mtbf300_durlognorm35_03.jspmu', 'la20_mu_mtbf300_durlognorm35_03.jspmu', 'orb01_mu_mtbf300_durlognorm35_03.jspmu', 'orb02_mu_mtbf300_durlognorm35_03.jspmu', 'orb03_mu_mtbf300_durlognorm35_03.jspmu', 'orb04_mu_mtbf300_durlognorm35_03.jspmu', 'orb05_mu_mtbf300_durlognorm35_03.jspmu', 'orb06_mu_mtbf300_durlognorm35_03.jspmu', 'orb07_mu_mtbf300_durlognorm35_03.jspmu', 'orb08_mu_mtbf300_durlognorm35_03.jspmu', 'orb09_mu_mtbf300_durlognorm35_03.jspmu', 'orb10_mu_mtbf300_durlognorm3

In [15]:
rules = {
    "Shortest_processing_time":'SPT',
    "Largest_processing_time":'LPT',
    "First_come_first_served":'FCFC',
    "Most_work_remaining":"MWR",
    "Most_operations_remaining":"MOR",
    "Random":'Random'
         }

In [33]:
test = JSPMUInstance.read(f'{extract_dir}/{file_names[8]}')

In [34]:
print(test)

Name: la05_mu_mtbf300_durlognorm35_03
n jobs: 10
n machines: 5
jobs:
[(1, 72), (0, 87), (4, 95), (2, 66), (3, 60)]
[(4, 5), (3, 35), (0, 48), (2, 39), (1, 54)]
[(1, 46), (3, 20), (2, 21), (0, 97), (4, 55)]
[(0, 59), (3, 19), (4, 46), (1, 34), (2, 37)]
[(4, 23), (2, 73), (3, 25), (1, 24), (0, 28)]
[(3, 28), (0, 45), (4, 5), (1, 78), (2, 83)]
[(0, 53), (3, 71), (1, 37), (4, 29), (2, 12)]
[(4, 12), (2, 87), (3, 33), (1, 55), (0, 38)]
[(2, 49), (3, 83), (1, 40), (0, 48), (4, 7)]
[(2, 65), (3, 17), (0, 90), (4, 27), (1, 23)]
Unavailabilities:
Machine 0 : [(306, 330), (382, 431), (1050, 1073), (1073, 1098), (1098, 1119), (1303, 1327), (1801, 1840), (1851, 1899), (1899, 1943), (2273, 2340)]
Machine 1 : [(241, 293), (834, 872), (872, 899), (899, 928), (971, 1008), (1110, 1137), (1938, 1979), (1994, 2033), (2137, 2174)]
Machine 2 : [(449, 478), (520, 545), (671, 721), (763, 788), (855, 906), (1008, 1042), (1042, 1062), (1493, 1525), (1638, 1679), (1731, 1767), (1902, 1922)]
Machine 3 : [(92, 12

In [45]:
data = []
problematics = []
feasibility = []

for instance_name in tqdm(file_names):
  id_instance = instance_name.split('_')[0]
  row = {'id':id_instance}
  try:
    jspmu_instance = JSPMUInstance.read(f'{extract_dir}/{instance_name}')
    machine_unavailability = jspmu_instance.machine_holes

    for dispatching_rule in DispatchingRuleType:
      r = dispatching_rule.capitalize()
      jsl_inst, dispatcher, schedule = solve_jspmu_with_dispatching(
          jspmu_instance,
          dispatching_rule=dispatching_rule,
      )

      feasible,violations = validate_jsl_schedule_against_jspmu_real_only(jspmu_instance, schedule)

      feasibility.append((id_instance,rules[r],feasible,violations))

      start_times, end_times, makespan = schedule_to_plot_inputs(schedule, n_original_jobs=None)

      heur = rules[r]
      row[f'{heur}'] = makespan
    data.append(row)
  except:
    for dispatching_rule in DispatchingRuleType:
      r = dispatching_rule.capitalize()
      heur = rules[r]
      row[f'{heur}'] = -1
    data.append(row)
    problematics.append(instance_name)

100%|██████████| 24/24 [00:00<00:00, 41.63it/s]


In [46]:
feasible_df = pd.DataFrame(feasibility, columns = ['instance','Heuristic','Valid','Violations'])
feasible_df

,instance,Heuristic,Valid,Violations
0,abz5,SPT,True,"{'machine_overlap': [], 'job_precedence': [], ..."
1,abz5,LPT,True,"{'machine_overlap': [], 'job_precedence': [], ..."
2,abz5,FCFC,True,"{'machine_overlap': [], 'job_precedence': [], ..."
3,abz5,MWR,True,"{'machine_overlap': [], 'job_precedence': [], ..."
4,abz5,MOR,True,"{'machine_overlap': [], 'job_precedence': [], ..."
...,...,...,...,...
133,orb10,LPT,True,"{'machine_overlap': [], 'job_precedence': [], ..."
134,orb10,FCFC,True,"{'machine_overlap': [], 'job_precedence': [], ..."
135,orb10,MWR,True,"{'machine_overlap': [], 'job_precedence': [], ..."
136,orb10,MOR,True,"{'machine_overlap': [], 'job_precedence': [], ..."


In [47]:
print(f'Invalid heuristic outputs = {len(feasible_df) - feasible_df['Valid'].sum()}')

Invalid heuristic outputs = 0


In [48]:
df = pd.DataFrame(data)

In [49]:
df

,id,SPT,LPT,FCFC,MWR,MOR,Random
0,abz5,6085,6090,2093,2198,7835,3630
1,abz6,4154,5370,1493,1724,5070,2327
2,ft06,-1,-1,-1,-1,-1,-1
3,ft10,3440,3848,1838,1863,4360,2568
4,la01,2135,2396,1215,1528,3159,1516
5,la02,2550,2585,1310,1328,3009,1898
6,la03,1625,2213,1220,1228,2129,1696
7,la04,2034,2046,1048,1445,2988,2034
8,la05,1916,2194,732,1111,2528,1475
9,la16,4221,3713,1799,1885,4658,2362


In [50]:
df.to_csv('constructive_heuristics_results.csv',index=False)